In [1]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

d:\program\anaconda3\envs\transformer\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1、加载数据集

In [2]:
import pandas as pd
# 加载数据
data_path = './nlp/data/news.csv'  # 上传的文件路径
news_data = pd.read_csv(data_path)

texts = news_data['text'].tolist()
labels = news_data['label'].tolist()

# 使用 LabelEncoder 将字符串标签转换为数值
label_encoder = LabelEncoder()
labels = label_encoder.fit_transform(labels)

# 使用 scikit-learn 进行数据集划分
train_texts, test_texts, train_labels, test_labels = train_test_split(texts, labels, test_size=0.2, random_state=42)


In [3]:
num_classes = news_data['label'].nunique()

In [4]:
# 定义设备
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## 2、加载模型和分词器

In [5]:
from transformers import BertTokenizer, BertForSequenceClassification,AutoModelForSequenceClassification
import torch

# 加载中文的教师模型和分词器
teacher_model = BertForSequenceClassification.from_pretrained('bert-base-chinese', cache_dir='./models', num_labels=num_classes)
tokenizer = BertTokenizer.from_pretrained('bert-base-chinese', cache_dir='./models')

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-chinese and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [20]:
from transformers import BertConfig, BertForSequenceClassification

# 定义学生模型的配置
student_config = BertConfig(
    hidden_size=720,  # 缩小模型的隐藏层维度
    num_hidden_layers=6,  # 减少层数
    num_attention_heads=6,  # 减少注意力头的数量
    intermediate_size=1024,
    num_labels=num_classes
)

# 创建学生模型
student_model = BertForSequenceClassification(student_config)


## 3、将数据集分词和编码

In [7]:
# 自定义Dataset类
class NewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        # 注意: 需要将编码后的input_ids和attention_mask从第一维度（batch维度）中取出
        input_ids = encoding['input_ids'].squeeze(0)
        attention_mask = encoding['attention_mask'].squeeze(0)
        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': torch.tensor(label, dtype=torch.long)
        }

In [8]:

# 构建训练集和测试集
train_dataset = NewsDataset(train_texts, train_labels, tokenizer)
test_dataset = NewsDataset(test_texts, test_labels, tokenizer)

# 使用DataLoader
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

In [21]:
teacher_model.to(device)
student_model.to(device)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 720, padding_idx=0)
      (position_embeddings): Embedding(512, 720)
      (token_type_embeddings): Embedding(2, 720)
      (LayerNorm): LayerNorm((720,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-5): 6 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=720, out_features=720, bias=True)
              (key): Linear(in_features=720, out_features=720, bias=True)
              (value): Linear(in_features=720, out_features=720, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=720, out_features=720, bias=True)
              (LayerNorm): LayerNorm((720,), eps=1e-1

## 4、 定义损失函数和优化器

In [22]:
import torch.nn.functional as F
from transformers import  AdamW
# 定义蒸馏损失函数
def distillation_loss(student_logits, teacher_logits, labels, alpha=0.5, temperature=2.0):
    # 计算软目标损失（KL散度）
    soft_loss = F.kl_div(
        F.log_softmax(student_logits / temperature, dim=-1),
        F.softmax(teacher_logits / temperature, dim=-1),
        reduction='batchmean'
    ) * (temperature ** 2)

    # 计算硬目标损失（交叉熵损失）
    hard_loss = F.cross_entropy(student_logits, labels)

    # 综合两部分损失
    return alpha * soft_loss + (1.0 - alpha) * hard_loss


# 定义优化器
teacher_optimizer = AdamW(teacher_model.parameters(), lr=5e-6)
student_optimizer = AdamW(student_model.parameters(), lr=5e-6)

## 5、 训练学生模型

In [11]:
import time
num_epochs = 10
def train_and_evaluate(model, optimizer, dataloader, num_epochs, device='cuda'):
    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        start_time = time.time()

        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            total_loss += loss.item()

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        avg_loss = total_loss / len(dataloader)
        end_time = time.time()

        # 验证
        model.eval()
        total_correct = 0
        with torch.no_grad():
            for batch in test_loader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['labels'].to(device)

                logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
                preds = torch.argmax(logits, dim=-1)
                total_correct += (preds == labels).sum().item()

        accuracy = total_correct / len(test_loader.dataset)
        print(f'Epoch {epoch + 1}/{num_epochs}, Train Loss: {avg_loss:.4f},Test Accuracy {epoch + 1}: {accuracy:.4f}, Time: {end_time - start_time:.2f} seconds')
    return model

In [12]:
import torch
import time

def train_student_with_distillation(student_model, teacher_model, train_dataloader, optimizer, device='cuda', num_epochs=3):
    teacher_model.eval()  # 教师模型在蒸馏时应该是固定的
    student_model.train()

    for epoch in range(num_epochs):
        total_loss = 0
        start_time = time.time()

        for batch in train_dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            # 获取教师模型的输出
            with torch.no_grad():
                teacher_logits = teacher_model(input_ids=input_ids, attention_mask=attention_mask).logits

            # 获取学生模型的输出
            student_logits = student_model(input_ids=input_ids, attention_mask=attention_mask).logits

            # 计算蒸馏损失
            loss = distillation_loss(student_logits, teacher_logits, labels, alpha=0.8, temperature=5.0)
            total_loss += loss.item()

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        avg_loss = total_loss / len(train_dataloader)
        end_time = time.time()
        # 在测试集上评估学生模型
        student_model.eval()
        total_correct = 0
        with torch.no_grad():
            for batch in test_loader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['labels'].to(device)

                student_logits = student_model(input_ids=input_ids, attention_mask=attention_mask).logits
                preds = torch.argmax(student_logits, dim=-1)
                total_correct += (preds == labels).sum().item()

        accuracy = total_correct / len(test_loader.dataset)
        print(f'Epoch {epoch + 1}/{num_epochs}, Train Loss: {avg_loss:.4f},Test Accuracy {epoch + 1}: {accuracy:.4f}, Time: {end_time - start_time:.2f} seconds')

    return student_model


In [13]:
# 训练教师模型
print("Training Teacher Model...")
trained_teacher_model = train_and_evaluate(teacher_model, teacher_optimizer, train_loader, num_epochs, device='cuda')


Training Teacher Model...
Epoch 1/10, Train Loss: 1.9819,Test Accuracy 1: 0.7900, Time: 8.71 seconds
Epoch 2/10, Train Loss: 1.2713,Test Accuracy 2: 0.9650, Time: 8.50 seconds
Epoch 3/10, Train Loss: 0.7665,Test Accuracy 3: 0.9800, Time: 8.47 seconds
Epoch 4/10, Train Loss: 0.4815,Test Accuracy 4: 0.9850, Time: 8.45 seconds
Epoch 5/10, Train Loss: 0.3162,Test Accuracy 5: 0.9850, Time: 8.50 seconds
Epoch 6/10, Train Loss: 0.2193,Test Accuracy 6: 0.9900, Time: 8.55 seconds
Epoch 7/10, Train Loss: 0.1587,Test Accuracy 7: 0.9850, Time: 8.49 seconds
Epoch 8/10, Train Loss: 0.1170,Test Accuracy 8: 0.9900, Time: 8.54 seconds
Epoch 9/10, Train Loss: 0.0900,Test Accuracy 9: 0.9900, Time: 8.52 seconds
Epoch 10/10, Train Loss: 0.0798,Test Accuracy 10: 0.9850, Time: 8.56 seconds


In [23]:

# 训练学生模型
print("\nTraining Student Model with Distillation...")
student_optimizer = AdamW(student_model.parameters(), lr=1e-5)
num_epochs = 20
trained_student_model = train_student_with_distillation(student_model, teacher_model, train_loader, student_optimizer, device='cuda', num_epochs=num_epochs)



Training Student Model with Distillation...
Epoch 1/20, Train Loss: 2.2064,Test Accuracy 1: 0.1250, Time: 5.55 seconds
Epoch 2/20, Train Loss: 2.1849,Test Accuracy 2: 0.1750, Time: 5.28 seconds
Epoch 3/20, Train Loss: 2.1547,Test Accuracy 3: 0.1250, Time: 5.78 seconds
Epoch 4/20, Train Loss: 2.1234,Test Accuracy 4: 0.3050, Time: 7.65 seconds
Epoch 5/20, Train Loss: 1.9991,Test Accuracy 5: 0.3850, Time: 7.62 seconds
Epoch 6/20, Train Loss: 1.6610,Test Accuracy 6: 0.7100, Time: 7.60 seconds
Epoch 7/20, Train Loss: 1.0909,Test Accuracy 7: 0.8000, Time: 7.66 seconds
Epoch 8/20, Train Loss: 0.7208,Test Accuracy 8: 0.8300, Time: 7.63 seconds
Epoch 9/20, Train Loss: 0.5068,Test Accuracy 9: 0.8750, Time: 7.64 seconds
Epoch 10/20, Train Loss: 0.3794,Test Accuracy 10: 0.8950, Time: 7.80 seconds
Epoch 11/20, Train Loss: 0.2756,Test Accuracy 11: 0.9050, Time: 7.72 seconds
Epoch 12/20, Train Loss: 0.2218,Test Accuracy 12: 0.9050, Time: 5.94 seconds
Epoch 13/20, Train Loss: 0.1690,Test Accuracy 13:

## 6、模型对比

In [24]:
teacher_params = sum(p.numel() for p in teacher_model.parameters())
student_params = sum(p.numel() for p in student_model.parameters())

print(f"Teacher Model Total Parameters: {teacher_params}")
print(f"Student Model Total Parameters: {student_params}")
print(f"Reduction: {(teacher_params - student_params) / teacher_params * 100:.2f}%")


Teacher Model Total Parameters: 102275338
Student Model Total Parameters: 44207674
Reduction: 56.78%


In [25]:
# 对比教师模型和学生模型的效果
def compare_models(teacher_model, student_model, dataloader, device='cuda'):
    teacher_model.eval()
    student_model.eval()
    teacher_correct = 0
    student_correct = 0

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            teacher_logits = teacher_model(input_ids=input_ids, attention_mask=attention_mask).logits
            student_logits = student_model(input_ids=input_ids, attention_mask=attention_mask).logits

            teacher_preds = torch.argmax(teacher_logits, dim=-1)
            student_preds = torch.argmax(student_logits, dim=-1)

            teacher_correct += (teacher_preds == labels).sum().item()
            student_correct += (student_preds == labels).sum().item()

    teacher_accuracy = teacher_correct / len(dataloader.dataset)
    student_accuracy = student_correct / len(dataloader.dataset)

    print(f'Teacher Model Accuracy: {teacher_accuracy:.4f}')
    print(f'Student Model Accuracy: {student_accuracy:.4f}')

In [26]:
# 对比效果
print("\nComparing Teacher and Student Models...")
compare_models(trained_teacher_model, trained_student_model, test_loader, device='cuda')


Comparing Teacher and Student Models...
Teacher Model Accuracy: 0.9850
Student Model Accuracy: 0.9250
